# Example: Iron(II) oxidation (decoupled Fe(II) from Fe(III) in the database)

Authors: G. Dan Miron

A chemical system was constructed in GEM-Selektor (see the GEMSprj folder). Fe(II) and Fe(III) were decoupled by introducing a new independent component (IComp) denoted Feiii. Using the database backup and restore functions, all occurrences of Fe(III) in species formulas were replaced with Feiii through a text-editor find-and-replace operation (|+3| and |3| replaced by iii).

This approach allows the Fe(II)/Fe(III) partitioning to be controlled externally through a kinetic rate function, while GEMS calculates equilibrium for the specified Fe(II) and Fe(III) composition at each step. In the present example, iron-bearing solid phases were suppressed to prevent precipitation, such that iron remains exclusively in the aqueous phase.

The chemical system composition, thermodynamic conditions, and rate constant values are provided solely for demonstration purposes. The system is in equilibirum with calcite and air (atmosphere) log(pCO2) -3.5

In [ ]:
## Import python packages
import numpy as np
import xgems as xg
import sys

## Setting global variables, paths

In [ ]:
# file name for system definitions
xg_system_filename='gems_files/Iron2Dec-dat.lst'

xgEngine = xg.ChemicalEngine(xg_system_filename)

## Defining the lists of indexes for elements, substance
- hint these can be kept in a separate python file an used in other projects

In [ ]:
# lists of elements and phases
element_symbols = ['Feiii', 'Fe', 'Ca', 'Na', 'Cl', 'C', 'O', 'Zz']
phase_names = {
    'Fe(OH)3(mic)': 'Fe(OH)3',
    'Fe(OH)2(s)': 'Fe(OH)2',
    'gas_gen': 'gas',
    'aq_gen': 'aqueous',
}

# dictionary of element names and index
element_indexes = {symbol: xgEngine.indexElement(symbol) for symbol in element_symbols}
# dictionary of phase names and index
phase_indexes = {name: xgEngine.indexPhase(name) for name in phase_names}

In [ ]:
# copy from leaching example, add Al, Mg
# Lists of what we will output 
output_aqueous_elements = ['Feiii', 'Fe']
output_properties = ['time', 'pH', 'gems_code','rate']

In [ ]:
# initialize the output containers, also dictionaries
aqueous_table = {element: [] for element in output_aqueous_elements}
properties_table = {prop: [] for prop in output_properties}

## Defining the kinetic function

In [ ]:
# writing a function that calculates the oxidation rate 
# cTau =: 10^(k)*(my[{OH-}])^2*(my[{O2(aq)}])*m_t[{Fe}]; 
# k - rate constant  mol^-3 kg^3 min^-1
# O2aq, OHminus, iron2 are concentrations of [O2(aq)], [OH-], and [Fe(II)] in mol/kg H2O 

def oxidation_rate(k,O2aq,OHminus,iron2):
    return np.power(10.0, k) * np.power(OHminus, 2) * O2aq * iron2

## Set global parameters, run test equilibration

In [ ]:
print('Script for iron oxidation with gems kernel')

# Initial equilibration
# take some parameters as read !
T = xgEngine.temperature()
P = xgEngine.pressure()
b = xgEngine.elementAmounts().copy() # the composition was set in GEM-Selektor system 

# now we suppress the precipitation of solids
xgEngine.setSpeciesUpperLimit("Fe(OH)3(mic)",0.0)
xgEngine.setSpeciesUpperLimit("Fe(OH)2(s)",0.0)


xgEngine.setColdStart()
code = xgEngine.equilibrate(T, P, b)
if code != 2 and code != 6:
            print("WARNING: Problem with GEMS output code at the beginning...... ")
            sys.exit()

# we store some common used indexes in own variables 
index_OHminus=xgEngine.indexSpecies('OH-')
index_O2aq=xgEngine.indexSpecies('O2(aq)')
index_Fe2=xgEngine.indexElement('Fe')
index_Fe3=xgEngine.indexElement('Feiii')
index_CO2g=xgEngine.indexSpecies('CO2(g)')

# some testing output
print('dcomp index for O2(aq)', index_O2aq,'OH-', index_OHminus) 
print('pH ', xgEngine.pH())

aq_volume = xgEngine.phaseVolume(phase_indexes['aq_gen']) * 1000  # m³ to L
aq_elements = xgEngine.elementAmountsInPhase(phase_indexes['aq_gen'])

Fe2_concentration =aq_elements[index_Fe2] / aq_volume
O2_concentration = xgEngine.speciesMolalities()[index_O2aq]
OHminus_concentration =  xgEngine.speciesMolalities()[index_OHminus]

print('iron oxidation rate', oxidation_rate(11.65, O2_concentration, OHminus_concentration, Fe2_concentration), 'mol/min')
print(Fe2_concentration)
print(O2_concentration)
print(OHminus_concentration)
print('log10(pCO2)',xgEngine.lnActivities()[index_CO2g]/np.log(10))

## Dissolution time loop

In [ ]:
# use SI units
# year_to_seconds=365.25*24.0*3600.0

# Temperature in Kelvin
T=25.0+273.15

# definitions for time discretization
tend=200 # 
delta_t=1.0  # 
time = 0.0 # start

time=time+delta_t # only for first time step
code = 2

k = 12.6 # mol^-3 kg^3 min^-1

# initial values moles
b[index_Fe2] = 0.001
b[index_Fe3] = 1e-15
code = xgEngine.equilibrate(T, P, b)

# while loop
while time <= tend :
    print('\rcalculate for time = '+str(time)+' minutes', 'dt ', delta_t, 'code', code, end='')    

    aq_volume = xgEngine.phaseVolume(phase_indexes['aq_gen']) * 1000  # m³ to L
    aq_elements = xgEngine.elementAmountsInPhase(phase_indexes['aq_gen'])

    Fe2_concentration =aq_elements[index_Fe2] / aq_volume
    O2_concentration = xgEngine.speciesMolalities()[index_O2aq]
    OHminus_concentration =  xgEngine.speciesMolalities()[index_OHminus]
    
    
    # we call the rate function
    drate = oxidation_rate(k,O2_concentration,OHminus_concentration,Fe2_concentration) 

    if (drate > b[index_Fe2]): 
        print('rate ',drate,' larger than Fe(II) ', b[index_Fe2],'\n')
        break

    # calculate the new Fe(II) and Fe(III) mol amounts 
    b[index_Fe2] = b[index_Fe2] - drate
    b[index_Fe3] = b[index_Fe3] + drate 

    
    # we run GEMS            
    xgEngine.setColdStart()
    code = xgEngine.equilibrate(T, P, b)

    # check if GEMS calculation was successful, if not try to improve numerics and reequilibrate
    if not (code == 2 or code == 6):  # 2 and 6 are  good solution ...3 is maybe good solution due to divergence problems
        print('t , dt ',time,' ',delta_t,' rescaled limits and run again\n')
        code =xgEngine.reequilibrate(False)
        if not (code == 2):
            print("Error: Problem with GEMS output code during equilibration table! at time returned code ",time,code)
            print("for detailed error diagnostics look into ipmlog.txt")
            print('t , dt ',time,' ',delta_t,' rate: ',drate,' amount per dt: ',drate*delta_t)
            print("bulk composition",xgEngine.elementAmounts())
            print("dll",xgEngine.speciesUpperLimits())
            time= 2*tend
            outflag = False
            break
    
    # properties
    properties_table['time'].append(time)
    properties_table['pH'].append(xgEngine.pH())
    properties_table['gems_code'].append(code)
    properties_table['rate'].append(drate)

    # Aqueous species concentrations
    for element in output_aqueous_elements:
        index = element_indexes[element]
        aqueous_table[element].append(1000*aq_elements[index] / aq_volume)


    # increase the time
    time=time+delta_t

print("\neverything finished")

In [ ]:
# copy from leaching example and change cycle to time
# plot the results
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'font.size': 15})

# Create plot
fig, ax1 = plt.subplots(figsize=(7, 5))

for elem in output_aqueous_elements:
    ax1.plot(properties_table['time'], aqueous_table[elem], label=elem)

ax1.set_xlabel('time (minutes)')
ax1.set_ylabel('total conc. in solution (mmol/l)')
ax1.set_title('iron oxidation')
ax1.grid(True)
#ax1.set_xscale('log')

ax2 = ax1.twinx()
ax2.plot(properties_table['time'], properties_table['pH'], label='pH', color='black', linestyle='--')
ax2.set_ylabel('pH')


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()